In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder \
   .appName("Working with data frames") \
   .getOrCreate()


In [ ]:
df=spark.read.csv("retail_transactions_100k.csv",header=True,inferSchema=True)
df.show()

In [ ]:
df2=df.drop("rating")
df2.show(5)

In [ ]:
df2=df.drop("customer_name","payment_method","rating")
df2.show(5)

In [ ]:
from pyspark.sql.functions import col
df2=df.filter(col("status")!="Cancelled")
df2.show(5)

In [ ]:
df2=df.dropDuplicates()
df2.show(5)

In [ ]:
df2=df.dropDuplicates(["city","region"])
df2.show(5)

In [ ]:
df2=df.distinct()
df2.show(5)

In [ ]:
df.select("city").distinct().show()

In [ ]:
#Drop columns having Null values
df2=df.na.drop()
df2.show()

In [ ]:
df2=df.na.drop(subset=["payment_method"])
df2.show(5)

In [ ]:
#Orderby and sort both does the same work
df.orderBy("unit_price").show(5)
#df.sort("unit_price").show(5) --> produces the same result

In [ ]:
df.orderBy(col("category").asc(), col("unit_price").desc()).show(5)

In [ ]:
#withColumn creates a new column
df2=df.withColumn("gross_amount",col("quantity")* col("unit_price"))
df2.orderBy(col("gross_amount").desc()).show(5)

**SORT OPERATIONS:**
1.Sort unit price ascending.
2.Sort unit price descending.
3.Sort quantity descending.
4.Sort by category and then unit price descending.
6.Create gross_amount and show top 10 orders
6.Sort by region and then city.
7.Sort by status and then quantity descending.
8.Display the 5 lowest-priced transactions.



In [ ]:
#1
df.orderBy(col("unit_price").asc()).show(5)

In [ ]:
#2
df.orderBy(col("unit_price").desc()).show(5)

In [ ]:
#3
df.orderBy(col("quantity").desc()).show(5)

In [ ]:
#4
df.orderBy(col("category").desc(),col("unit_price").desc()).show(5)

In [ ]:
#5
df2=df2.withColumn("gross_amount",col("unit_price")*col("quantity"))
df2.show(10)

In [ ]:
#6
df.orderBy(col("region"),col("city")).show(5)

In [ ]:
#7
df.orderBy(col("status"),col("quantity").desc()).show(5)

In [ ]:
#8
df.orderBy(col("unit_price").asc()).show(5)

ARITHMETIC OPERATIONS

In [ ]:
from pyspark.sql.functions import sum,avg,min,max,count

In [ ]:
df.count()

In [ ]:
df.select(sum("quantity").alias("total_quantity")).show()

In [ ]:
df.select(avg("unit_price").alias("Average_price")).show()

In [ ]:
df.select(max("unit_price").alias("Highest_price")).show()

In [ ]:
df.select(count("*").alias("Total_records"),
          sum("quantity").alias("Total_quantity"),
          avg("unit_price").alias("Average_price"),
          min("unit_price").alias("Lowest_price"),
          max("unit_price").alias("Highest_price")).show()

Arithmetic operations-TASKS
1. Create gross_amount.
2. Find total gross sales.
3. Create discount_amount.
4. Find total discount given.
5. Create final_amount.
6. Find total final sales.
7. Find total sales from Delivered orders only.
8. Find the average value of Delivered orders.
9. Count Electronics transactions.
10. Count Cancelled transactions.

In [ ]:
#1
df2=df.withColumn("gross_amount",col("unit_price")*col("quantity"))
#2
df2.select(sum("gross_amount").alias("Total_gross")).show()

In [ ]:
#3
df2=df.withColumn("discount_amount",col("discount_pct")/100 *  col("unit_price"))
df2.show(5)

In [ ]:
#4
df2=df2.withColumn("total_discount_given",
                  col("unit_price")-col("discount_amount"))
df2.show(5)

In [ ]:
#5
df2=df2.withColumn("Final_amount",col("gross_amount")-col("discount_amount"))
#6
df2.select(sum("Final_amount").alias("Total_final_sales")).show()

In [ ]:
#7
df2.filter(col("status")=="Delivered") \
.select(sum("Final_amount").alias("total_delivered_sales")) \
.show()

In [ ]:
#8
df2.filter(col("status")=="Delivered") \
.select(avg("Final_amount").alias("Average_of_deliverd_orders")).show()

In [ ]:
#9
df2.filter(col("category")=="Electronics") \
 .select(count("*").alias("Count_of_Electronics_Transactions")) \
  .show()

In [ ]:
#10
df2.filter(col("status")=="Cancelled") \
.select(count("*").alias("Cancelled_Transactions")) \
.show()

Arithmetic operations using groupBy

In [ ]:
df.groupBy("category") \
.count() \
.show()

In [ ]:
df.groupBy("status") \
.count() \
.show()

In [ ]:
df.groupBy("category") \
.agg(
    sum("quantity").alias("total_quantity")
) \
.show()

In [ ]:
df.groupBy("category") \
.agg(
    count("*").alias("Total_orders"),
    sum("quantity").alias("Total_quantity"),
    avg("unit_price").alias("Average_price"),
    min("unit_price").alias("Lowest_price"),
    max("unit_price").alias("Highest_price")) \
    .show()

JOIN OPERATIONS


In [ ]:
customers_data = [
    ("C00001", "Aman Retail", "Hyderabad", "Corporate"),
    ("C00002", "Blue Mart", "Mumbai", "Retail"),
    ("C00003", "Fresh Basket", "Bangalore", "Retail"),
    ("C00004", "Prime Stores", "Delhi", "SMB"),
    ("C00005", "Quick Shop", "Pune", "Corporate")
]

customers_columns = [
    "customer_id",
    "customer_name_master",
    "customer_city",
    "segment"
]

customers_df = spark.createDataFrame(
    customers_data,
    customers_columns
)

customers_df.show()

In [ ]:
joined_df=df.join(customers_df,on="customer_id",how="inner")
joined_df.show()

In [ ]:
left_join_df=df.join(customers_df,on="customer_id",how="left")
left_join_df.show()

In [ ]:
right_join_df=df.join(customers_df,on="customer_id",how="right")
right_join_df.show()

In [ ]:
full_join_df=df.join(customers_df,on="customer_id",how="outer")
full_join_df.show()

**WINDOW FUNCTION** - Splitting and ranking. First split into windows and then rank it

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    row_number,
    rank,
    dense_rank,
    sum,
    avg,
    lag,
    lead
)

In [ ]:
product_df=df.groupBy("Category","product_name") \
.agg(avg("unit_price").alias("avg_price"))
product_df.show()

In [ ]:
window_spec = Window.partitionBy("category") \
    .orderBy(col("avg_price").desc())

ranked_products = product_df.withColumn(
    "rank",
    row_number().over(window_spec)
)

ranked_products.show()

DATES AND YEARS

In [ ]:
from pyspark.sql.functions import col, to_date

df2 = df.withColumn(
    "order_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)

In [ ]:
from pyspark.sql.functions import year
df2.withColumn("year",year("order_date")).select("order_date","year").show()

In [ ]:
from pyspark.sql.functions import month
df2.withColumn("month",month("order_date")).select("order_date","month").show()

In [ ]:
from pyspark.sql.functions import dayofmonth
df2.withColumn("day",dayofmonth("order_date")).select("order_date","day").show()